In [20]:
# 데이터 생성
import pandas as pd
dataset = pd.read_csv('../../data/csv/movie_reviews2.csv')

In [21]:
# 독립, 종속 분리
X = dataset['review'].to_numpy()
y = dataset['label'].to_numpy().astype('float32')

In [22]:
# 텍스트 전처리


# 형태소 분석
from konlpy.tag import Okt
okt = Okt()
# ----------------------------------------------------------------
# 문장을 형태소 별로 분리한 후, 공백으로 구분하여 다시 한 문장으로 만드는 함수
# ----------------------------------------------------------------
def tokenize_korean(text_list):
  return [" ".join(okt.morphs(text, stem=True)) for text in text_list]
korean_morphs = tokenize_korean(X)



In [23]:
# 벡터화
from tensorflow.keras import layers
vectorize_layer = layers.TextVectorization(
	max_tokens=1000, # 벡터화 할 때 사용할 최대 단어 수
 	output_mode='int', # 벡터 수치를 정수로 변환
	output_sequence_length=10 # 문장 길이를 10으로 자동 고정
)
vectorize_layer.adapt(korean_morphs)

In [24]:
# 파이프라인 적용(웬만하면 하는게 좋음)
# X를 학습시킬 때 문자열 이슈로 인해 에러가 발생
# 파이프라인을 적용하면 텐서플로우의 텐서 객체로 변환되기 때문에 이슈가 해결
import tensorflow as tf
# from_tensor_slices : korean_morphs와 y에서 하나씩 꺼내 하나의 세트로 묶음
train_ds = tf.data.Dataset.from_tensor_slices((korean_morphs, y)).batch(8)

In [25]:
from tensorflow.keras import models

def build_transform_model():
  # 입력층
  inputs = layers.Input((1,),dtype=tf.string)
  
  # 백터화와 임베딩
  x = vectorize_layer(inputs)
  vocab_size = vectorize_layer.vocabulary_size()
  x = layers.Embedding(input_dim=vocab_size, output_dim=64)(x)
  
  # 멀티헤드어텐션
  attention_output = layers.MultiHeadAttention(num_heads=2,key_dim=64)(x,x)
  
  # 잔차 연결 및 정규화
  # 잔차 연결 : 어텐션 결과에 원래 입력값을 더해 정보를 보존
  x = layers.Add()([x,attention_output])
  x = layers.LayerNormalization()(x)
  
  # 추가학습
  ffn = layers.Dense(64, activation='relu')(x)
  
  # 잔차 연결 및 정규화
  x = layers.Add()([x,ffn])
  x = layers.LayerNormalization()(x)
  
  # 출력층
  x = layers.GlobalAveragePooling1D()(x)
  outputs = layers.Dense(1,activation='sigmoid')(x)
  return models.Model(inputs = inputs, outputs = outputs)

# 모델 설계
model = build_transform_model()



In [26]:
# 모델 설정
model.compile(optimizer='adam',loss='binary_crossentropy',metrics=['accuracy'])

In [27]:
# 학습
model.fit(train_ds, epochs=50)

Epoch 1/50
4/4 ━━━━━━━━━━━━━━━━━━━━ 3s 6ms/step - accuracy: 0.6333 - loss: 0.6852
Epoch 2/50
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.7333 - loss: 0.5736 
Epoch 3/50
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.7333 - loss: 0.4672 
Epoch 4/50
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.8667 - loss: 0.4026 
Epoch 5/50
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.8667 - loss: 0.3317
Epoch 6/50
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.9333 - loss: 0.2683
Epoch 7/50
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.9333 - loss: 0.2157
Epoch 8/50
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.9333 - loss: 0.1713
Epoch 9/50
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.9333 - loss: 0.1380
Epoch 10/50
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.9333 - loss: 0.1183
Epoch 11/50
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.9333 - loss: 0.1088
Epoch 12/50
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.9333 - loss: 0.1078
Epoch 13/5

In [31]:
# 테스트
test_text = ["영화가 너무 재미있어서 하나도 안 지루해요"]

# 형태소별로 문장을 수정
test_morphs = tokenize_korean(test_text)

# 테스트 할 때 텐서 타입을 변환(문자열 이슈)
test_input = tf.constant(test_morphs)

# 예측 실행
predictions = model.predict(test_input)
predictions_size = len(predictions)
for i in range(predictions_size):
  p = predictions[i][0] * 100
  print(f'{test_text[i]}:{'긍정' if p > 50 else '부정'}({p:.2f}%)')

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 37ms/step
영화가 너무 재미있어서 하나도 안 지루해요:부정(49.39%)
